# 0 — Introduction

## Measuring grounding in a LoRA-fine-tuned language model

### The problem

Suppose you fine-tune a small model to interpret poems, and it returns this:

> *The poem reflects on the passage of time and the way memory shapes experience. It uses natural imagery to suggest change, and contrasts light and darkness to mark shifts in feeling. The tone is reflective and somewhat melancholy.*

That reads like a competent interpretation. It is also true of an enormous number of poems, and it would fit most of this corpus unchanged. The model may not have engaged with the particular poem in front of it at all — it may have learned what interpretations *sound like*.

**This failure is not invisible.** Put that text next to the actual poem and a careful reader would often notice it says nothing specific. It is worth being clear about that, because the interesting difficulty is not detection:

- there are **750 outputs** to check — 150 poems × 5 model variants — and each needs reading against its source poem
- "too generic" is a judgement call with no obvious cut-off, and two people will draw the line differently
- **it produces no number**, so two models cannot be compared, and there is no way to say whether fine-tuning helped or hurt

So the problem is not that this failure hides. It is that it is **unmeasured** — and an unmeasured failure cannot be compared across models, tracked as you change things, reported, or defended.

This project builds that measurement, then applies it to a model it fine-tunes.

## Why it is worth the effort

Fine-tuning on synthetic data — targets generated by a larger model — is now the common way to adapt a small model, because expert-written targets are expensive and generated ones are not. Gudibande et al. (2023) studied models trained this way and found they *"are adept at mimicking ChatGPT's style but not its factuality"*: human raters scored them well, because style is the part that transfers most readily.

That was a different setting — larger models, a different task, human raters rather than a mechanical check — so it does not predict what happens here. It does make the question worth asking rather than assuming, which is the whole reason this project measures instead of eyeballing.

Poetry interpretation is the **testbed, not the application**. It was chosen for three practical reasons:

| Property | Why it matters |
|---|---|
| public domain | the text can be printed, committed, and sent to third-party APIs without a copyright problem |
| short | a poem and its interpretation fit a small model's context window |
| quotable | interpretations are supposed to quote their source, which gives a mechanical check most generation tasks do not offer |

The last one is what makes the task usable at all. If an interpretation claims the poem contains a line, that claim can be checked by substring match — no model, no judge, no opinion. The evaluation built here would work on any task with that property.

## The central measurement: the swap test

Take an interpretation `I` that a model produced for poem `P`. Show a judge that same interpretation three times, against three different poems.

| Condition | Poem shown | Question it asks |
|---|---|---|
| `matched` | `P` itself | — |
| `mismatched_random` | random poem, **different author** | does this text fit *any* poem? |
| `mismatched_same_author` | different poem, **same author** | does it fit any poem *by this poet*? |

Two numbers come out:

- **Grounding gap** = matched − mismatched_random
- **Poem-level gap** = matched − mismatched_same_author

A small gap means the output is interpretation-shaped text that fits anything — the template above would score a gap near zero. A large gap means it is anchored to the specific poem in front of it.

**Why the third condition.** An author's themes recur across their work. A model that learned *"Dickinson writes about death, immortality and nature"* can produce a plausible interpretation of an unseen Dickinson poem **without reading it** — and that text still beats a random Whitman poem, so the standard control scores it as well grounded. Only comparing against *another Dickinson poem* closes that route. The difference between the two gaps is the share of apparent grounding that is really just recognising the poet.

### Why this works without a correct answer

There is no reference "correct interpretation" to compare against, and no expert available to write two thousand of them. Most evaluation designs stall here.

The swap test sidesteps it by being **relative**. A judge that scores generously, harshly, or inconsistently does so in *both* conditions, so the bias appears on both sides of the subtraction and cancels. What survives is the part that depends on which poem was shown — which is exactly the quantity of interest.

This is the single design decision the project rests on, so it is validated before it is trusted: on day 2 the swap test is run on the **teacher's** outputs, where grounding is expected. If the judges cannot separate matched from mismatched there, the evaluation is invalid and the design changes before any effort is spent on training.

## Experimental design: five arms

One model is not an experiment. Five arms are compared, and each pairwise contrast isolates exactly one mechanism.

| Arm | Mechanism | What it is |
|---|---|---|
| `template` | none | one fixed generic interpretation, emitted for every poem |
| `base_zero` | none | base model, zero-shot |
| `base_few` | in-context | base model, 3 examples in the prompt |
| `lora_r8` | weight update | LoRA, rank 8 |
| `lora_r16` | weight update | LoRA, rank 16 |

| Contrast | Question |
|---|---|
| `template` vs `base_few` | does the model beat a trivial generic baseline at all? |
| `base_zero` vs `base_few` | what do in-context examples contribute? |
| `base_few` vs `lora_r8` | what do weight updates add over prompting? — **the headline** |
| `lora_r8` vs `lora_r16` | what does adapter capacity contribute? |

`template` is a single block of interpretation-shaped text that never sees a poem — it has a central idea, images, a tone and an interpretive claim, so it satisfies the format check while containing nothing specific to anything. Whatever it scores is **how much of a "good" score is available for free**, without reading. If it scores well, that is a real and interesting result rather than a bug: it sets the floor every other arm has to clear.

## What gets measured

**Model-free first**, because these cost nothing and cannot be argued with:

| Metric | Question |
|---|---|
| format compliance | does the output follow the required four-part schema? |
| grounding rate | do quoted lines actually appear in the poem? (substring match) |
| length distribution | do arms differ merely in verbosity? |

**Then judge-based**, with two judges from different model families:

| Metric | Question |
|---|---|
| pairwise win rate vs `base_few` | with binomial confidence intervals |
| position-bias flip rate | every comparison run twice, order swapped |
| swap test | grounding gap and poem-level gap per arm |
| inter-judge agreement (Cohen's κ) | is the instrument itself reliable? |

Hypotheses are **pre-registered** — written down with their statistical tests fixed before any result exists. Nulls are reported as plainly as positives, and a null means *"no detected difference"*, never *"proof of equality"*.

## Design decisions and the constraints that set them

Every number below is either measured or derived. Where an earlier version of this project used a figure chosen by argument, measuring it changed the figure — so the working rule is that a threshold has to come from somewhere checkable.

### The model stack

| Role | Model | The constraint it satisfies |
|---|---|---|
| Student | Qwen2.5-0.5B **base** | small enough for 19 runs in 30 GPU-hours; 32K context; clean `q_proj`/`k_proj`/`v_proj` for LoRA |
| Teacher | DeepSeek | competent at the task, ~$1–2 for the corpus, distinct family |
| Judge (primary) | GPT-4o-mini | closest to the family Zheng et al. 2023 validated LLM-judging with |
| Judge (secondary) | Gemini 2.5 Flash | distinct from all three above |

**Base, not Instruct — this one is load-bearing.** An instruction-tuned checkpoint already emits structured output. If the student arrived knowing how to produce four numbered sections, format compliance would start at ceiling and H1 would measure nothing. The model was chosen to make a measurement *possible*, not because it was convenient.

**Four distinct families, for a cited reason.** Evaluators recognise and favour their own generations (Panickssery et al. 2024), which imposes three constraints at once: teacher ≠ judge, judge ≠ judge, student ≠ judge. Qwen / DeepSeek / OpenAI / Google satisfies all three.

### The sequence budget: why 2048

Qwen handles 32,768 tokens. The limit here is the GPU, not the model — and Kaggle's T4 (Turing) and P100 (Pascal) predate FlashAttention-2, so attention materialises the full `batch × heads × seq × seq` score matrix.

| `MAX_SEQ_LEN` | attention memory / layer | usable corpus |
|---|---|---|
| 1024 | 0.23 GB | 78.6% |
| **2048** | **0.94 GB** | **89.4%** |
| 4096 | 3.76 GB | 94.3% |
| 8192 | 15.03 GB | *does not fit* |

At batch 8 with gradient checkpointing, 2048 leaves room for weights, gradients and optimiser state on a 16 GB card. Doubling to 4096 buys **five percentage points of corpus for four times the attention memory**; 8192 is not a trade-off at all, since it cannot run.

The cap is also not binding on most examples: the median training sequence is **517 tokens** and the 99th percentile is **1,746**. Raising the ceiling would help a thin tail while every batch pays the memory reservation.

This is hardware-contingent rather than a claim about the task. On an A100 with FlashAttention the memory term is linear instead of quadratic, and the whole calculation changes.

### Poem length: derived, not chosen

An earlier version capped poems at 100 lines. Measuring showed that nothing under ~150 lines systematically exceeds the token budget, and that **line count is a poor proxy for token count** — a 193-line poem of short lines fits, while a 120-line poem of long ones does not. The cap was rejecting on the wrong variable and discarding ~110 usable poems, 4.4% of the corpus.

The upper bound is now arithmetic:

```
MAX_POEM_TOKENS = MAX_SEQ_LEN − PROMPT_OVERHEAD_TOKENS = 2048 − 416 = 1632
```

whatever remains of the sequence budget once the prompt and a maximum-length interpretation are accounted for. The lower bound stays a line count — 8 lines — because it is about having enough *distinct* lines to quote two or three without reproducing the poem.

**What this excludes.** 11% of poems clearing the minimum are still too long to train on, and they are systematically the longest ones. The corpus is therefore biased toward shorter work, which belongs alongside the other sampling biases rather than being passed over.

## What this project does not claim

Stated here, at the front, rather than buried at the end. A limitation a reader discovers is worth less than one you named yourself — and for a project whose entire argument is *measure honestly rather than assume*, hiding the limits would undercut the thing it is arguing.

- **Nothing about interpretation quality.** Only about textual grounding. There is no expert reference, so *"is this a good reading of the poem"* is outside what anything here can support.
- **The training targets are synthetic.** The student's ceiling is the teacher's quality — so the teacher's hallucination rate is measured and reported, not assumed away.
- **The base model was pretrained on this corpus.** Public-domain poetry is public domain *because* it is old and widely reproduced, which is exactly what puts it in web-scale pretraining data. A contamination probe measures how much of the corpus the base model reproduces verbatim, and results are stratified by it — but this is measured, not solved. No absolute grounding rate is claimed anywhere; every claim is a comparison between arms that share one base model and therefore share its contamination.
- **Two judges, neither validated against human annotation** on this task. Agreement between two language models is weaker evidence than agreement with a person would be.
- **The corpus is bounded by a compute budget, not by anything about poetry.** 11% of poems clearing the minimum length are too long to train on within `MAX_SEQ_LEN`, and they are systematically the longest. Results describe shorter work; the bound would move on different hardware.
- **One task, one backbone, one teacher.** Nothing here generalises beyond this setup.

## How to read the rest

| Notebook | Contents |
|---|---|
| `01_data` | where the poems come from, how interpretations are generated, the filtering funnel, and the author-grouped fold split |
| `02_training` | LoRA setup, loss masking, the sweep, and what crosses to the GPU |
| `03_evaluation` | model-free metrics, both judges, the swap test, contamination |
| `04_report` | results, hypothesis outcomes, comparison to prior work, limitations |

**Logic lives in `src/` as tested functions.** These notebooks import, call and display — they contain no implementation. That is deliberate: it is what makes the pipeline testable, and it means every number shown here comes from code with a test beside it.

The configuration below is printed at the top of every notebook, so the settings a result was produced under are always recorded next to the result.

In [1]:
# Jupyter starts its kernel in notebooks/, so the project root has to be on
# the path before `import config` will work. Walking up to find config.py keeps
# this correct no matter where the notebook is launched from — and it is a
# no-op once the package is installed with `pip install -e .`
import sys
from pathlib import Path

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import config

print(config.summary())

SMOKE          : False
IS_KAGGLE      : False
seed           : 42

student model  : Qwen/Qwen2.5-0.5B
teacher model  : deepseek-chat
judge (primary): gpt-4o-mini
judge (2nd)    : gemini-2.5-flash  [robustness only]

corpus cap     : none (all survivors), >= 8 lines, <= 1632 poem tokens
interpretation : 80-250 words
max seq len    : 2048 tokens (drop, never truncate)

folds          : 5, grouped by author
eval poems     : 30 per fold = 150 total

lora rank      : 8 (alpha 16)
target modules : q_proj, k_proj, v_proj
max steps      : 1000 (fixed steps, not epochs)
batch size     : 8 x 2 accum

data dir       : /Users/adelinchaushev/Desktop/PoetryIntepretations./data
results dir    : /Users/adelinchaushev/Desktop/PoetryIntepretations./results
